In [ ]:

# Experiment 9: Perceptron vs Multilayer Perceptron (A/B Experiment)
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from PIL import Image

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, roc_curve, auc,
                              classification_report)

import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
np.random.seed(42)


In [ ]:

# Loaded the English Handwritten Characters Dataset
data_path = "English Handwritten Characters Dataset"

labels_df = pd.read_csv(f"{data_path}/english.csv")
print(labels_df.shape)
labels_df.head()


In [ ]:

# Read and preprocess all images
# Resize to a small fixed size, convert to grayscale, flatten, normalize
img_size = 32

X = []
y = []

for _, row in labels_df.iterrows():
    img_path = os.path.join(data_path, row["image"])
    img = Image.open(img_path).convert("L")
    img = img.resize((img_size, img_size))
    arr = np.array(img, dtype=np.float32) / 255.0
    X.append(arr.flatten())
    y.append(row["label"])

X = np.array(X)
y = np.array(y)
print("Data shape:", X.shape)
print("Number of classes:", len(np.unique(y)))


In [ ]:

# Encode string labels into integers
le = LabelEncoder()
y_encoded = le.fit_transform(y)
n_classes = len(le.classes_)
print("Classes:", n_classes)


In [ ]:

# Class distribution check
plt.figure(figsize=(12,4))
sns.countplot(x=y)
plt.xticks(rotation=90)
plt.title("Class distribution across 62 characters")
plt.tight_layout()
plt.show()


In [ ]:

# Train test split, stratified since many classes have few samples
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
print(X_train.shape, X_test.shape)


In [ ]:

# Model A: Single Layer Perceptron Learning Algorithm, implemented from scratch
# Multiclass is handled using one vs rest binary perceptrons
class PerceptronScratch:
    def __init__(self, n_features, n_classes, lr=0.01, epochs=50):
        self.lr = lr
        self.epochs = epochs
        self.n_classes = n_classes
        # one weight vector and bias per class
        self.W = np.zeros((n_classes, n_features))
        self.b = np.zeros(n_classes)
        self.errors_per_epoch = []

    def step(self, x):
        return np.where(x >= 0, 1, 0)

    def fit(self, X, y):
        n_samples = X.shape[0]
        for epoch in range(self.epochs):
            total_errors = 0
            for i in range(n_samples):
                xi = X[i]
                for c in range(self.n_classes):
                    target = 1 if y[i] == c else 0
                    score = np.dot(self.W[c], xi) + self.b[c]
                    pred = self.step(score)
                    error = target - pred
                    if error != 0:
                        self.W[c] += self.lr * error * xi
                        self.b[c] += self.lr * error
                        total_errors += 1
            self.errors_per_epoch.append(total_errors)
        return self

    def decision_scores(self, X):
        return X.dot(self.W.T) + self.b

    def predict(self, X):
        scores = self.decision_scores(X)
        return np.argmax(scores, axis=1)


In [ ]:

# Train the perceptron on a reduced epoch count since it runs sample by sample
pla = PerceptronScratch(n_features=X_train.shape[1], n_classes=n_classes, lr=0.01, epochs=20)
pla.fit(X_train, y_train)


In [ ]:

# Plot PLA training error vs epochs
plt.figure(figsize=(6,4))
plt.plot(range(1, len(pla.errors_per_epoch)+1), pla.errors_per_epoch, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Total misclassifications")
plt.title("PLA Training Error vs Epochs")
plt.tight_layout()
plt.show()


In [ ]:

# Evaluate PLA on test data
y_pred_pla = pla.predict(X_test)

pla_acc = accuracy_score(y_test, y_pred_pla)
pla_prec = precision_score(y_test, y_pred_pla, average="macro", zero_division=0)
pla_rec = recall_score(y_test, y_pred_pla, average="macro", zero_division=0)
pla_f1 = f1_score(y_test, y_pred_pla, average="macro", zero_division=0)

print("PLA Accuracy:", pla_acc)
print("PLA Precision (macro):", pla_prec)
print("PLA Recall (macro):", pla_rec)
print("PLA F1 (macro):", pla_f1)


In [ ]:

# Confusion matrix for PLA, shown as heatmap without class names since 62 classes
cm_pla = confusion_matrix(y_test, y_pred_pla)
plt.figure(figsize=(8,6))
sns.heatmap(cm_pla, cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix: PLA")
plt.tight_layout()
plt.show()


In [ ]:

# Model B: Multilayer Perceptron
# Hyperparameter tuning over activation, optimizer, learning rate, hidden layers, batch size
param_grid = {
    "hidden_layer_sizes": [(64,), (128,64), (128,64,32)],
    "activation": ["relu", "tanh"],
    "solver": ["sgd", "adam"],
    "learning_rate_init": [0.001, 0.01],
    "batch_size": [32, 64]
}

base_mlp = MLPClassifier(max_iter=200, random_state=42, early_stopping=True)

grid_search = GridSearchCV(base_mlp, param_grid, cv=3, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV accuracy:", grid_search.best_score_)


In [ ]:

# Train final MLP using the best hyperparameters found
best_mlp = grid_search.best_estimator_
y_pred_mlp = best_mlp.predict(X_test)

mlp_acc = accuracy_score(y_test, y_pred_mlp)
mlp_prec = precision_score(y_test, y_pred_mlp, average="macro", zero_division=0)
mlp_rec = recall_score(y_test, y_pred_mlp, average="macro", zero_division=0)
mlp_f1 = f1_score(y_test, y_pred_mlp, average="macro", zero_division=0)

print("MLP Accuracy:", mlp_acc)
print("MLP Precision (macro):", mlp_prec)
print("MLP Recall (macro):", mlp_rec)
print("MLP F1 (macro):", mlp_f1)


In [ ]:

# Plot MLP training loss curve to see convergence
plt.figure(figsize=(6,4))
plt.plot(best_mlp.loss_curve_)
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title("MLP Training Loss Curve")
plt.tight_layout()
plt.show()


In [ ]:

# Confusion matrix for MLP
cm_mlp = confusion_matrix(y_test, y_pred_mlp)
plt.figure(figsize=(8,6))
sns.heatmap(cm_mlp, cmap="Greens")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix: MLP")
plt.tight_layout()
plt.show()


In [ ]:

# ROC curves, one vs rest, micro and macro average
y_test_bin = label_binarize(y_test, classes=range(n_classes))

# Scores for PLA come from decision function, scores for MLP come from predict_proba
pla_scores = pla.decision_scores(X_test)
mlp_scores = best_mlp.predict_proba(X_test)

def micro_macro_roc(y_true_bin, scores, n_classes):
    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], scores[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    fpr_micro, tpr_micro, _ = roc_curve(y_true_bin.ravel(), scores.ravel())
    auc_micro = auc(fpr_micro, tpr_micro)

    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes
    auc_macro = auc(all_fpr, mean_tpr)

    return fpr_micro, tpr_micro, auc_micro, all_fpr, mean_tpr, auc_macro

pla_fpr_mi, pla_tpr_mi, pla_auc_mi, pla_fpr_ma, pla_tpr_ma, pla_auc_ma = micro_macro_roc(y_test_bin, pla_scores, n_classes)
mlp_fpr_mi, mlp_tpr_mi, mlp_auc_mi, mlp_fpr_ma, mlp_tpr_ma, mlp_auc_ma = micro_macro_roc(y_test_bin, mlp_scores, n_classes)


In [ ]:

# Plot ROC curves for both models
plt.figure(figsize=(7,6))
plt.plot(pla_fpr_mi, pla_tpr_mi, label=f"PLA micro (AUC={pla_auc_mi:.2f})")
plt.plot(pla_fpr_ma, pla_tpr_ma, label=f"PLA macro (AUC={pla_auc_ma:.2f})")
plt.plot(mlp_fpr_mi, mlp_tpr_mi, label=f"MLP micro (AUC={mlp_auc_mi:.2f})")
plt.plot(mlp_fpr_ma, mlp_tpr_ma, label=f"MLP macro (AUC={mlp_auc_ma:.2f})")
plt.plot([0,1],[0,1],"k--", linewidth=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves: PLA vs MLP")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

# A/B comparison table
comparison = pd.DataFrame({
    "Model": ["PLA", "MLP"],
    "Accuracy": [pla_acc, mlp_acc],
    "Precision": [pla_prec, mlp_prec],
    "Recall": [pla_rec, mlp_rec],
    "F1 Score": [pla_f1, mlp_f1],
    "ROC AUC (micro)": [pla_auc_mi, mlp_auc_mi],
    "ROC AUC (macro)": [pla_auc_ma, mlp_auc_ma]
})
comparison


In [ ]:

# Bar plot comparing PLA and MLP across metrics
metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1 Score"]
comparison_melted = comparison.melt(id_vars="Model", value_vars=metrics_to_plot,
                                     var_name="Metric", value_name="Score")

plt.figure(figsize=(8,5))
sns.barplot(x="Metric", y="Score", hue="Model", data=comparison_melted)
plt.title("PLA vs MLP: Metric Comparison")
plt.tight_layout()
plt.show()


In [ ]:

# Summary of findings
print("Best MLP hyperparameters:", grid_search.best_params_)
print()
print(comparison)
print()
print("PLA final epoch errors:", pla.errors_per_epoch[-1])
print("MLP number of iterations run:", best_mlp.n_iter_)
